## 1. Imports

In [ ]:
# Cell 1  -  Imports
import os
import json
import hashlib
from pathlib import Path

import faiss
import numpy as np
import tiktoken
import pdfplumber
import docx

from openai import OpenAI

## 2. Environment & Client Setup

In [ ]:
# Cell 2 Load env + init client
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 3. File Parsing 

In [ ]:
# Cell 3  -  File parsing (file_utils logic)
def parse_pdf(path: str) -> str:
    text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text.append(content)
    return "\n".join(text)


def parse_docx(path: str) -> str:
    doc = docx.Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())


def parse_txt(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")


def parse_file(path: str) -> str:
    """Dispatch to the correct parser based on file extension."""
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return parse_pdf(path)
    elif ext == ".docx":
        return parse_docx(path)
    elif ext == ".txt":
        return parse_txt(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")


# Quick smoke test  -  swap in a real file path to try it
# print(parse_file("../uploads/sample.pdf")[:500])

## 4 Text Chunking

In [ ]:
# Cell 4  -  Text chunking
EMBED_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 500    # tokens per chunk
CHUNK_OVERLAP = 50  # tokens shared between adjacent chunks

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping token-based chunks."""
    enc = tiktoken.encoding_for_model(EMBED_MODEL)
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        if end >= len(tokens):
            break
        start += chunk_size - overlap  # slide forward, keeping overlap
    return chunks

# Quick check
sample_text = "word " * 1200  # ~1200 tokens
chunks = chunk_text(sample_text)
print(f"{len(chunks)} chunks  -  first chunk token count: {len(tiktoken.encoding_for_model(EMBED_MODEL).encode(chunks[0]))}")

## 5. Embedding, Index, Retrieval 

In [ ]:
# Cell 5  -  Embedding, FAISS index, retrieval
def file_hash(path: str) -> str:
    """MD5 hash of file contents  -  used as cache key."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def embed_chunks(chunks: list[str]) -> np.ndarray:
    """Embed a list of text chunks. Returns float32 array of shape (n, dims)."""
    response = client.embeddings.create(model=EMBED_MODEL, input=chunks)
    vectors = [item.embedding for item in response.data]
    return np.array(vectors, dtype=np.float32)


def build_index(embeddings: np.ndarray) -> faiss.IndexFlatL2:
    """Build a flat L2 FAISS index from an embedding matrix."""
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index


def retrieve(query: str, index: faiss.IndexFlatL2, chunks: list[str], k: int = 4) -> list[str]:
    """Embed query and return the top-k most relevant chunks."""
    query_vec = embed_chunks([query])          # shape (1, dims)
    _, indices = index.search(query_vec, k)   # indices shape (1, k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]


# Embedding cache: avoids re-embedding a file that hasn't changed
# { file_hash: {"chunks": [...], "index": faiss.Index} }
embed_cache: dict = {}

## 6. Chat history trimming

In [ ]:
# Cell 6  -  Chat history trimming (memory_utils logic)
CHAT_MODEL = "gpt-4o-mini"
MAX_HISTORY_TOKENS = 3000  # budget for history; leaves room for system prompt + context + response

def count_tokens(messages: list[dict], model: str = CHAT_MODEL) -> int:
    """Count tokens used by a list of chat messages."""
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # role + framing overhead per message
        total += len(enc.encode(msg["content"]))
    total += 2  # reply priming overhead
    return total


def trim_history(messages: list[dict], max_tokens: int = MAX_HISTORY_TOKENS) -> list[dict]:
    """Drop oldest messages (excluding system prompt) until within token budget."""
    system = [m for m in messages if m["role"] == "system"]
    history = [m for m in messages if m["role"] != "system"]

    while history and count_tokens(system + history) > max_tokens:
        history.pop(0)  # drop oldest non-system message

    return system + history


# Smoke test
dummy_messages = [{"role": "system", "content": "You are a helpful assistant."}]
dummy_messages += [{"role": "user", "content": f"Message {i} " + "token " * 100} for i in range(20)]
trimmed = trim_history(dummy_messages)
print(f"Before: {len(dummy_messages)} messages | After trim: {len(trimmed)} messages | Tokens: {count_tokens(trimmed)}")

## 7. Prompt builder + chat completion 

In [ ]:
# Cell 7 - Prompt builder + chat completion
SYSTEM_PROMPT = (
    "You are a helpful assistant. "
    "When relevant document context is provided, use it to answer accurately. "
    "If the context does not cover the question, say so and answer from general knowledge."
)

def build_messages(history: list[dict], user_query: str, context_chunks: list[str]) -> list[dict]:
    """Assemble the message list to send to the chat API."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if context_chunks:
        sep = "\n\n---\n\n"
        context_text = sep.join(context_chunks)
        rag_block = "Relevant document context:\n\n" + context_text
        messages.append({"role": "system", "content": rag_block})

    messages += history
    messages.append({"role": "user", "content": user_query})
    return messages


def chat(user_query: str, history: list[dict], context_chunks: list[str] = []) -> str:
    """Single round-trip to the chat API. Returns the assistant reply string."""
    messages = build_messages(history, user_query, context_chunks)
    messages = trim_history(messages)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.3,
    )
    return response.choices[0].message.content


# Smoke test (makes a real API call)
# reply = chat("What is RAG in AI?", history=[])
# print(reply)


## 8. Document Ingest

In [ ]:
# Cell 8  -  Document ingest (parse -> chunk -> embed -> cache)
def ingest_file(path: str) -> tuple[list[str], faiss.IndexFlatL2]:
    """
    Full ingest pipeline for a single file.
    Returns (chunks, index). Uses embed_cache to skip re-embedding unchanged files.
    """
    fh = file_hash(path)
    if fh in embed_cache:
        print(f"Cache hit for {Path(path).name}  -  skipping embedding")
        return embed_cache[fh]["chunks"], embed_cache[fh]["index"]

    print(f"Ingesting {Path(path).name}...")
    text = parse_file(path)
    chunks = chunk_text(text)
    print(f"  {len(chunks)} chunks created")

    embeddings = embed_chunks(chunks)
    index = build_index(embeddings)
    print(f"  Embeddings done  -  index has {index.ntotal} vectors")

    embed_cache[fh] = {"chunks": chunks, "index": index}
    return chunks, index


## 9. End-to-end RAG loop

In [ ]:
# Cell 9 - End-to-end RAG loop
def rag_chat(user_query: str, history: list[dict], path: str | None = None) -> tuple[str, list[dict]]:
    """
    Full RAG round-trip.
    - If a file path is given, ingest it (cached) and retrieve relevant chunks.
    - Returns reply and a new history list (does not mutate the input).
    """
    context_chunks = []
    if path:
        chunks, index = ingest_file(path)
        context_chunks = retrieve(user_query, index, chunks, k=4)

    reply = chat(user_query, history, context_chunks)

    updated_history = history.copy()
    updated_history.append({"role": "user", "content": user_query})
    updated_history.append({"role": "assistant", "content": reply})

    return reply, updated_history


# Interactive smoke test
# history = []
# reply, history = rag_chat("Summarise the main points", history, path="../uploads/sample.pdf")
# print(reply)
# reply, history = rag_chat("What does it say about X?", history, path="../uploads/sample.pdf")
# print(reply)
